In [1]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

pd.set_option("display.max_columns", None)

df = pd.read_csv(r"C:\Users\thana\Desktop\flight_analysis\data\flight_ml_ready.csv")

print("Γραμμές:", len(df))
print("Έτοιμο!")

Γραμμές: 1928371
Έτοιμο!


In [4]:
# Καθαρισμός NaN στο DEP_PERIOD
df["DEP_PERIOD"] = df["DEP_PERIOD"].fillna("unkwonn")

# Κατηγορικές στήλες
df["Month_cat"] = "Month_" + df["Month"].astype(str)
df["DayType"] = df["IS_WEEKEND"].map({0: "Weekday", 1: "Weekend"})
df["Delayed_cat"] = df["DELAYED"].map({0: "OnTime", 1: "Delayed"})

# Δημιουργία transactions
transactions = df[["UniqueCarrier", "Month_cat", "DEP_PERIOD",
                   "DayType", "Delayed_cat"]].values.tolist()

print("Transactions ready!")
print("Παράδειγμα:", transactions[0])

Transactions ready!
Παράδειγμα: ['WN', 'Month_1', 'evening', 'Weekday', 'OnTime']


In [5]:
# Encoding
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

#FP-Growth
frequent_itemsets = fpgrowth(df_encoded, min_support=0.05, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False)

print("Frequent Itemsets:", len(frequent_itemsets))
print(frequent_itemsets.head(10))

Frequent Itemsets: 79
     support                         itemsets
0   0.737011             frozenset({Weekday})
1   0.569336              frozenset({OnTime})
6   0.452469           frozenset({afternoon})
7   0.430664             frozenset({Delayed})
27  0.417674     frozenset({OnTime, Weekday})
48  0.330937  frozenset({afternoon, Weekday})
51  0.319337    frozenset({Delayed, Weekday})
5   0.268770             frozenset({morning})
2   0.268400             frozenset({evening})
8   0.262989             frozenset({Weekend})


In [7]:
# Association rules
rules = association_rules(frequent_itemsets, metric="confidence",
                          min_threshold=0.6,
                          num_itemsets=len(frequent_itemsets))
rules = rules.sort_values("lift", ascending=False)

print("Rules:", len(rules))
print(rules[["antecedents", "consequents", "support",
             "confidence", "lift"]].head(10))

Rules: 31
                      antecedents           consequents   support  confidence  \
25     frozenset({afternoon, WN})   frozenset({OnTime})  0.061205    0.686174   
9                 frozenset({WN})   frozenset({OnTime})  0.132588    0.679631   
14       frozenset({Weekday, WN})   frozenset({OnTime})  0.098907    0.679190   
6            frozenset({morning})   frozenset({OnTime})  0.166089    0.617958   
11  frozenset({morning, Weekday})   frozenset({OnTime})  0.122745    0.615997   
18           frozenset({Month_2})  frozenset({Weekday})  0.075517    0.772034   
26           frozenset({Month_5})  frozenset({Weekday})  0.060980    0.770030   
20           frozenset({Month_1})  frozenset({Weekday})  0.072976    0.769346   
29                frozenset({MQ})  frozenset({Weekday})  0.055527    0.758212   
16          frozenset({Month_12})  frozenset({Weekday})  0.078487    0.751097   

        lift  
25  1.205218  
9   1.193726  
14  1.192950  
6   1.085402  
11  1.081958  
18  1.04

## Ανάλυση Association Rules

### Κύρια συμπεράσματα

**Southwest Airlines(WN):**
- {OnTime} με confidence 67.9% και lift 1.19
- Εμφανίζεται συστηματικά ως πιο συνεπής εταιρεία

**Περίοδος ημέρας:**
- {morning} -> {OnTime} με confidence 61.8%
- Οι πρωινές πτήσεις είναι πιο συνεπείς

**Συνδυασμός {afternoon, WN} -> {OnTime}:**
- Το υψηλότερο lift: 1.21
- Οι απογευματινές πτήσεις της Southwest είναι αξιόπιστες

**Weekday pattern:**
- Οι περισσότεροι μήνες συσχετίζονται με Weekday
- Οι καθημερινές έχουν τις περισσότερες πτήσεις